# Data Collection (FBref)
---------------------------
Collects footballer season stats from FBref for the Top 5 leagues across 3 seasons (2022/23, 2023/24, 2024/25) using the ```soccerdata``` library.

Saves:
- ```data/raw/fbref/<league>_<season>_<stat_type>.csv``` (individual tables)
- ```data/processed/fbref_all.csv``` (merged, combined)

In [1]:
# Importing libraries

import time
import warnings
from pathlib import Path

import pandas as pd
import soccerdata as sd

warnings.filterwarnings('ignore')

[04/29/26 23:25:28] INFO     No custom team name replacements found. You can configure these in       ]8;id=9106843;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=9106844;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_config.py#92\92]8;;\
                             C:\Users\Nitro\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=9106850;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=9106851;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_config.py#190\190]8;;\
                             C:\Users\Nitro\soccerdata\config\league_dict.json.                                    

## Configuration

In [2]:
# Configuration

# Football leagues
LEAGUES = [
    'ENG-Premier League',
    'ESP-La Liga',
    'GER-Bundesliga',
    'ITA-Serie A',
    'FRA-Ligue 1'
]

# Seasons
SEASONS = ['2223', '2324', '2425']

# Stat types to include - each is a separate FBref table
STAT_TYPES = [
    'standard', # goals, assists, xG, xA
    'keeper', # saves, goals against, clean sheets
    'shooting', # shots, shots on target, xG per shot
    'playing_time', # minutes, appearances, starts
    'misc' # aerial duels won, fouls, cards
]

RAW_DIR = Path('../data/raw/fbref')
PROCESSED_DIR = Path('../data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Helpers

In [3]:
# Football league short names
LEAGUE_SHORT = {
    'ENG-Premier League': 'EPL',
    'ESP-La Liga': 'LaLiga',
    'GER-Bundesliga': 'Bundesliga',
    'ITA-Serie A': 'SerieA',
    'FRA-Ligue 1': 'Ligue1'
}

In [4]:
def fetch_stat(league: str, season: str, stat_type: str) -> pd.DataFrame | None:
    '''Fetch a single stat table and return it as a DataFrame, or None on failure.'''
    try:
        fbref = sd.FBref(leagues=league, seasons=season)
        df = fbref.read_player_season_stats(stat_type=stat_type)
        df = df.reset_index()

        # Tag with metadata
        df['league'] = LEAGUE_SHORT[league]
        df['season'] = season

        print(f'{LEAGUE_SHORT[league]} {season} {stat_type} - {len(df)} rows')
        return df

    except Exception as e:
        print(f'{LEAGUE_SHORT[league]} {season} {stat_type} - ERROR {e}')
        return None

## Collection Loop

In [5]:
all_standard = [] # Keep standard separately to anchor the merge
supplementary = {} # Keyed by stat_type

for league in LEAGUES:
    for season in SEASONS:
        print(f'\n{LEAGUE_SHORT[league]} {season}')

        # Standard table
        std = fetch_stat(league, season, 'standard')
        if std is not None:
            out_path = RAW_DIR / f'{LEAGUE_SHORT[league]}_{season}_standard.csv'
            std.to_csv(out_path, index=False)
            all_standard.append(std)
        
        # Supplementary tables
        for stat_type in STAT_TYPES[1:]: # Skip standard, as it is already done
            df = fetch_stat(league, season, stat_type)
            if df is not None:
                out_path = RAW_DIR / f'{LEAGUE_SHORT[league]}_{season}_{stat_type}.csv'
                std.to_csv(out_path, index=False)
                key = (league, season, stat_type)
                supplementary[key] = df
        
        time.sleep(3)


EPL 2223


                    INFO     Saving cached data to C:\Users\Nitro\soccerdata\data\FBref              ]8;id=9106858;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9106859;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

EPL 2223 standard - 569 rows


[04/29/26 23:25:35] INFO     Saving cached data to C:\Users\Nitro\soccerdata\data\FBref              ]8;id=9106864;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9106865;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

EPL 2223 keeper - 39 rows


...

Ligue1 2425 shooting - 553 rows


[04/29/26 23:47:28] INFO     Saving cached data to C:\Users\Nitro\soccerdata\data\FBref              ]8;id=9107315;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9107316;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Ligue1 2425 playing_time - 666 rows


[04/29/26 23:47:45] INFO     Saving cached data to C:\Users\Nitro\soccerdata\data\FBref              ]8;id=9107321;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=9107322;file://c:\Users\Nitro\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Ligue1 2425 misc - 553 rows


## Merging All Tables

In [42]:
if not all_standard:
    raise RuntimeError('No standard tables collected — check your connection/soccerdata install.')
 
# Combine all standard rows
base = pd.concat(all_standard, ignore_index=True)
print(f'Base (standard) shape: {base.shape}')

Base (standard) shape: (8595, 25)


In [43]:
# Identify merge keys - soccerdata uses MultiIndex, reset gives these columns
# Typical cols: player, nation, pos, squad, age, league, season, ...
MERGE_KEYS = ['player', 'squad', 'league', 'season']
 
# Columns to DROP from supplementary before merging (avoid duplicates)
# Keep only the stats columns - drop metadata that's already in base
DROP_COLS = ['nation', 'pos', 'age', 'born', '90s']

In [44]:
def safe_merge(base: pd.DataFrame, supp: pd.DataFrame, stat_type: str) -> pd.DataFrame:
    '''Merge a supplementary table into base, suffixing overlapping columns.'''
    # Drop redundant metadata cols if present
    drop = [c for c in DROP_COLS if c in supp.columns]
    supp = supp.drop(columns=drop + ['league', 'season'], errors='ignore')
 
    # Flatten MultiIndex columns if present
    if isinstance(supp.columns, pd.MultiIndex):
        supp.columns = ['_'.join(filter(None, map(str, col))).strip() for col in supp.columns]
 
    merged = base.merge(supp, on=['player', 'team'], how='left', suffixes=('', f'_{stat_type}'))
    return merged

In [45]:
# Flatten base MultiIndex columns first
if isinstance(base.columns, pd.MultiIndex):
    base.columns = ['_'.join(filter(None, map(str, col))).strip() for col in base.columns]

In [46]:
# Merge each supplementary type
for stat_type in STAT_TYPES[1:]:
    # Gather all rows for this stat_type across leagues/seasons
    frames = [
        v for k, v in supplementary.items() if k[2] == stat_type
    ]
    if not frames:
        print(f'Skipping {stat_type} - no data collected')
        continue
 
    supp_all = pd.concat(frames, ignore_index=True)
 
    # Flatten columns
    if isinstance(supp_all.columns, pd.MultiIndex):
        supp_all.columns = ['_'.join(filter(None, map(str, col))).strip() for col in supp_all.columns]
 
    base = safe_merge(base, supp_all, stat_type)
    print(f'After merging {stat_type}: {base.shape}')

After merging keeper: (9167, 44)
After merging shooting: (17675, 54)
After merging playing_time: (41059, 71)
After merging misc: (103921, 83)


## Data Cleaning

In [47]:
# Drop fully duplicate columns
base = base.loc[:, ~base.columns.duplicated()]

In [ ]:
# Filter: keep only players with meaningful playing time (>=90 mins)
before = len(base)
base = base[pd.to_numeric(base['Playing Time_Min'], errors='coerce').fillna(0) >= 90].copy()
print(f'\nFiltered by {'Playing Time_Min'} >= 90 mins: {before} -> {len(base)} rows')


Filtered by Playing Time_Min >= 90 mins: 103921 -> 97641 rows


In [52]:
base = base.reset_index(drop=True)

out_path = PROCESSED_DIR / 'fbref_all.csv'
base.to_csv(out_path, index=False)

print(f'\nDone - {len(base)} player-seasons saved to {out_path}')
print(f'Shape: {base.shape}')
print(f'Leagues: {base['league'].value_counts().to_dict()}')
print(f'Seasons: {base['season'].value_counts().to_dict()}')


Done - 97641 player-seasons saved to ..\data\processed\fbref_all.csv
Shape: (97641, 83)
Leagues: {'EPL': 23310, 'LaLiga': 21432, 'Bundesliga': 18312, 'SerieA': 18294, 'Ligue1': 16293}
Seasons: {'2324': 34998, '2223': 31558, '2425': 31085}
